# SkinSense: An Environment-Aware Skincare Recommendation Engine
https://world.openbeautyfacts.org/data

Aarya Nenawatee



Steps:
Cell 1  Install libraries

Cell 2  Load the dataset using openbeauty API

Cell 3  Clean ingredients (INCI map — no translation API needed)

Cell 4  Ingredient classification table

Cell 5  Skin quiz (user input)

Cell 6  Fetch weather using openweather API

Cell 7  Recommendation engine

Cell 8  Display results as dashboard



## Cell 1 Install libraries

In [ ]:
!pip install requests pandas langdetect -q
print(' All libraries ready')


## Cell 2  Load small sample from beauty dataset
> Keeping only rows where **ingredients is not null**.

In [ ]:
import os
if os.path.exists("beauty_data.jsonl.gz"):
    print("File already exists skipping download")
else:
    print("Downloading")
    !wget -c https://static.openbeautyfacts.org/data/openbeautyfacts-products.jsonl.gz -O beauty_data.jsonl.gz
    print("Download complete!")

In [ ]:
import gzip
import json
import pandas as pd

#  Products to KEEP — face and skin only
FACE_KEYWORDS = [
    "face", "facial", "skin care", "skincare",
    "moisturizer", "face cream", "face wash", "face serum",
    "face mask", "face toner", "face cleanser", "face lotion",
    "face oil", "face scrub", "face gel", "face mist",
    "eye cream", "lip balm", "sunscreen", "spf",
    "anti-aging", "anti-acne", "pore", "serum", "toner",
    "bb cream", "cc cream", "primer", "foundation base"
]

# Products to REJECT even if they pass above
REJECT_KEYWORDS = [
    "hair", "shampoo", "conditioner", "scalp",
    "body lotion", "body wash", "body cream", "body scrub",
    "hand cream", "hand wash", "hand lotion", "foot cream",
    "nail", "deodorant", "shower", "bath", "soap bar",
    "baby", "intimate", "feminine"
]

rows    = []
scanned = 0
skipped_no_name        = 0
skipped_no_ingredients = 0
skipped_not_face       = 0
skipped_body_hair      = 0

print("Loading full dataset — face/skin products only...")
print("Progress updates every 50,000 lines.\n")

with gzip.open('beauty_data.jsonl.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        scanned += 1

        if scanned % 50000 == 0:
            print(f"  Scanned {scanned:,} lines | Kept {len(rows):,} products...")

        try:
            p = json.loads(line)

            name        = (p.get('product_name') or '').strip()
            ingredients = (p.get('ingredients_text') or '').strip()
            category    = (p.get('categories') or '').strip().lower()
            name_lower  = name.lower()

            # Rule 1: skip if name empty
            if not name:
                skipped_no_name += 1
                continue

            # Rule 2: skip if ingredients null/empty
            if not ingredients:
                skipped_no_ingredients += 1
                continue

            # Rule 3: must match face/skin keyword in category OR name  ← changed
            is_face = any(kw in category or kw in name_lower for kw in FACE_KEYWORDS)
            if not is_face:
                skipped_not_face += 1
                continue

            # Rule 4: reject hair, body, hand products  ← new
            is_body_hair = any(kw in category or kw in name_lower for kw in REJECT_KEYWORDS)
            if is_body_hair:
                skipped_body_hair += 1
                continue

            rows.append({
                'name'       : name,
                'brand'      : (p.get('brands') or '').strip(),
                'category'   : category[:120],
                'ingredients': ingredients,
                'labels'     : (p.get('labels') or '').strip(),
            })

        except (json.JSONDecodeError, KeyError):
            continue

df = pd.DataFrame(rows)

print(f"\n{'='*48}")
print(f"  DONE — Face/skin products only")
print(f"{'='*48}")
print(f"  Total lines scanned      : {scanned:,}")
print(f"  Face/skin products kept  : {len(df):,}")
print(f"  Skipped (no name)        : {skipped_no_name:,}")
print(f"  Skipped (no ingredients) : {skipped_no_ingredients:,}")
print(f"  Skipped (not face/skin)  : {skipped_not_face:,}")
print(f"  Skipped (hair/body/hand) : {skipped_body_hair:,}")
print(f"  Null ingredients         : {df['ingredients'].isnull().sum()}")
print(f"  Empty ingredients        : {(df['ingredients'] == '').sum()}")

df.head(5)


## Cell 3 Clean ingredients without any translation API needed
> **Why no googletrans?**  
> Beauty ingredients follow the **INCI standard** (International Nomenclature of Cosmetic Ingredients).  
> INCI names are in Latin/English by definition — aqua, glycerin, parfum — so we just need  
> a small lookup table to map the handful of non-English words to their English equivalents.  
> This works offline, instantly, and never breaks.

In [ ]:
import re
from langdetect import detect, LangDetectException
import requests as req
import time

# INCI map: ingredient-level non-English to English
INCI_MAP = {
    'eau': 'water', 'wasser': 'water', 'agua': 'water', 'acqua': 'water',
    'glycérine': 'glycerin', 'glicerina': 'glycerin', 'glycerol': 'glycerin',
    'parfum': 'fragrance', 'duftstoffe': 'fragrance', 'perfume': 'fragrance',
    'alcool': 'alcohol denat', 'alkohol': 'alcohol denat',
    'huile': 'oil', 'öl': 'oil', 'aceite': 'oil', 'olio': 'oil',
    'extrait': 'extract', 'extracto': 'extract', 'estratto': 'extract',
    'beurre': 'butter', 'burro': 'butter', 'beurre de karité': 'shea butter',
    'acide': 'acid', 'säure': 'acid', 'ácido': 'acid',
    'colorant': 'color', 'farbstoff': 'color', 'colorante': 'color',
    'conservateur': 'preservative',
}

# Category word map
CATEGORY_MAP = {
    'crème': 'cream',       'creme': 'cream',         'crema': 'cream',
    'visage': 'face',       'gesicht': 'face',         'cara': 'face',
    'hydratant': 'moisturizing',                       'idratante': 'moisturizing',
    'nettoyant': 'cleanser','reiniger': 'cleanser',    'limpiador': 'cleanser',
    'soin': 'care',         'pflege': 'care',          'cuidado': 'care',
    'peau': 'skin',         'haut': 'skin',            'piel': 'skin',
    'sérum': 'serum',       'suero': 'serum',
    'masque': 'mask',       'maske': 'mask',            'mascarilla': 'mask',
    'solaire': 'sun care',  'sonnenschutz': 'sun care',
    'lait': 'lotion',       'loción': 'lotion',
    'mousse': 'foam',       'espuma': 'foam',
    'contour': 'eye contour','yeux': 'eyes',           'augen': 'eyes',
    'anti-rides': 'anti-wrinkle',                      'anti-falten': 'anti-wrinkle',
    'huile': 'oil',         'aceite': 'oil',
    'baume': 'balm',        'bálsamo': 'balm',
    'exfoliant': 'exfoliator',                         'peeling': 'exfoliator',
}

# India cities with realistic weather profiles
INDIA_CITIES = {
    'Delhi':      {'temperature': 38, 'humidity': 45, 'uv_estimate': 8,  'aqi': 168, 'description': 'Hazy sunshine'},
    'Mumbai':     {'temperature': 32, 'humidity': 82, 'uv_estimate': 6,  'aqi': 94,  'description': 'Humid and cloudy'},
    'Bangalore':  {'temperature': 26, 'humidity': 68, 'uv_estimate': 5,  'aqi': 62,  'description': 'Partly cloudy'},
    'Chennai':    {'temperature': 34, 'humidity': 78, 'uv_estimate': 7,  'aqi': 88,  'description': 'Hot and humid'},
    'Kolkata':    {'temperature': 36, 'humidity': 72, 'uv_estimate': 7,  'aqi': 132, 'description': 'Hazy'},
    'Hyderabad':  {'temperature': 33, 'humidity': 55, 'uv_estimate': 7,  'aqi': 75,  'description': 'Mostly sunny'},
    'Pune':       {'temperature': 30, 'humidity': 60, 'uv_estimate': 6,  'aqi': 58,  'description': 'Pleasant'},
    'Jaipur':     {'temperature': 40, 'humidity': 30, 'uv_estimate': 9,  'aqi': 145, 'description': 'Hot and dry'},
    'Ahmedabad':  {'temperature': 39, 'humidity': 35, 'uv_estimate': 9,  'aqi': 120, 'description': 'Very hot'},
    'Lucknow':    {'temperature': 37, 'humidity': 48, 'uv_estimate': 8,  'aqi': 155, 'description': 'Sunny and hot'},
    'Chandigarh': {'temperature': 34, 'humidity': 42, 'uv_estimate': 7,  'aqi': 72,  'description': 'Sunny'},
    'Kochi':      {'temperature': 30, 'humidity': 88, 'uv_estimate': 5,  'aqi': 48,  'description': 'Overcast'},
    'Bhopal':     {'temperature': 35, 'humidity': 50, 'uv_estimate': 8,  'aqi': 98,  'description': 'Sunny'},
    'Nagpur':     {'temperature': 38, 'humidity': 40, 'uv_estimate': 8,  'aqi': 85,  'description': 'Hot'},
    'Indore':     {'temperature': 36, 'humidity': 45, 'uv_estimate': 8,  'aqi': 90,  'description': 'Sunny and warm'},
    'Patna':      {'temperature': 37, 'humidity': 60, 'uv_estimate': 8,  'aqi': 160, 'description': 'Humid and hazy'},
    'Surat':      {'temperature': 33, 'humidity': 70, 'uv_estimate': 7,  'aqi': 95,  'description': 'Humid'},
    'Vizag':      {'temperature': 31, 'humidity': 75, 'uv_estimate': 6,  'aqi': 55,  'description': 'Coastal breeze'},
}


# Helper functions
def has_non_ascii(text):
    return bool(re.search(r'[^\x00-\x7F]', str(text)))

def is_non_latin_script(text):
    """True if text has Cyrillic, Arabic, CJK — drop these rows."""
    return bool(re.search(
        r'[\u0400-\u04FF\u0600-\u06FF\u4E00-\u9FFF\u3040-\u30FF\uAC00-\uD7AF]',
        str(text)
    ))

def detect_lang(text):
    """
     FIX: detect actual language code before calling MyMemory.
    langdetect returns codes like 'fr', 'de', 'es' — NOT 'auto'.
    MyMemory needs explicit codes like 'fr|en'.
    """
    try:
        return detect(str(text))
    except LangDetectException:
        return 'fr'   # beauty products most often French if detection fails

def translate_mymemory(text, target='en'):
    """
    Translate using MyMemory free API — no key needed, 1000 calls/day.
    FIX 1: uses detect_lang() not 'auto' as source language
    FIX 2: skips API if already English
    FIX 3: catches error messages in response body
    """
    if not text or not has_non_ascii(text):
        return text   # already ASCII-safe

    source = detect_lang(text)

    if source == 'en':
        return text   # skip API call entirely

    try:
        url    = 'https://api.mymemory.translated.net/get'
        params = {'q': text[:200], 'langpair': f'{source}|{target}'}
        r      = req.get(url, params=params, timeout=8)
        r.raise_for_status()
        result     = r.json()
        translated = result.get('responseData', {}).get('translatedText', text)

        #  FIX 3: catch error strings returned in the response body
        if not translated or 'invalid' in str(translated).lower() or 'langpair' in str(translated).lower():
            return text

        return translated.lower().strip()

    except Exception:
        return text   # always fall back silently


def translate_column_batched(series, batch_size=10, delay=0.4):
    """Translate only rows with non-ASCII text. Shows progress."""
    result       = series.copy()
    to_translate = series[series.apply(has_non_ascii)]

    if len(to_translate) == 0:
        print('   No rows needed translation.')
        return result

    print(f'   Rows to translate: {len(to_translate)}')
    indices = to_translate.index.tolist()

    for i in range(0, len(indices), batch_size):
        batch = indices[i: i + batch_size]
        for idx in batch:
            result[idx] = translate_mymemory(series[idx])
        done = min(i + batch_size, len(indices))
        print(f'   Translated {done}/{len(indices)}...', end='\r')
        time.sleep(delay)

    print('\n   Done.')
    return result


def clean_ingredients(raw_text):
    """Clean raw ingredient string → list of English ingredient names."""
    text = str(raw_text).lower().strip()
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'\d+\.?\d*\s*%', '', text)
    text = re.sub(r'\s+', ' ', text)
    parts   = re.split(r'[,;]', text)
    cleaned = []
    for part in parts:
        part = part.strip().strip('.')
        if len(part) < 2:
            continue
        part = INCI_MAP.get(part, part)
        cleaned.append(part)
    return cleaned

def map_category(text):
    """Apply category word map offline — no API needed."""
    text = str(text).lower()
    for foreign, english in CATEGORY_MAP.items():
        text = text.replace(foreign, english)
    return text.strip()


# Apply cleaning
df['ingredients_clean'] = df['ingredients'].apply(clean_ingredients)
df['ingredients_str']   = df['ingredients_clean'].apply(lambda x: ', '.join(x))
df['category_en']       = df['category'].apply(map_category)

# Dropping non-Latin script rows (Cyrillic, Arabic, CJK)
before = len(df)
df = df[~df['name'].apply(is_non_latin_script)].copy()
df = df[~df['category'].apply(is_non_latin_script)].copy()
print(f'Dropped {before - len(df)} non-Latin rows | {len(df)} remaining')

# Translating name and brand using fixed MyMemory function
print('\nTranslating product names...')
df['name_en']  = translate_column_batched(df['name'])
print('Translating brand names...')
df['brand_en'] = translate_column_batched(df['brand'])

# Summary
print(f'\n Cleaning complete: {len(df)} rows')
print(f'   Available cities  : {list(INDIA_CITIES.keys())}')
print(f'\n🔹 Sample:')
df[['name_en', 'brand_en', 'category_en', 'ingredients_str']].head(5)



## Cell 4 Ingredient classification table
> Reference table: safe / harmful / comedogenic per skin type.

In [ ]:
data = [
    # ingredient            safety          suitable_for          description
    ['water',               'safe',         'all',                'base, always safe'],
    ['aqua',                'safe',         'all',                'water (INCI)'],
    ['glycerin',            'safe',         'all',                'humectant, draws moisture'],
    ['glycerol',            'safe',         'all',                'same as glycerin'],
    ['hyaluronic acid',     'safe',         'all',                'deep hydration'],
    ['sodium hyaluronate',  'safe',         'all',                'smaller hyaluronic acid'],
    ['niacinamide',         'safe',         'oily,acne',          'red/uces oil, pore-minimising'],
    ['ceramide',            'safe',         'dry,sensitive',      'strengthens skin barrier'],
    ['ceramide np',         'safe',         'dry,sensitive',      'specific ceramide type'],
    ['aloe vera',           'safe',         'sensitive,acne',     'soothing, anti-inflammatory'],
    ['aloe barbadensis',    'safe',         'sensitive,acne',     'aloe vera scientific name'],
    ['panthenol',           'safe',         'dry,sensitive',      'vitamin B5, calming'],
    ['zinc oxide',          'safe',         'acne,sensitive',     'mineral sunscreen'],
    ['titanium dioxide',    'safe',         'sensitive',          'mineral sunscreen'],
    ['salicylic acid',      'safe',         'acne,oily',          'exfoliates inside pores'],
    ['lactic acid',         'safe',         'dry',                'gentle exfoliation'],
    ['glycolic acid',       'safe',         'acne,dull',          'surface exfoliation'],
    ['squalane',            'safe',         'all',                'lightweight, non-comedogenic'],
    ['green tea extract',   'safe',         'acne,sensitive',     'antioxidant'],
    ['tocopherol',          'safe',         'dry,sensitive',      'vitamin E, antioxidant'],
    ['retinol',             'safe',         'aging',              'night use only, avoid high UV'],
    ['peptides',            'safe',         'aging',              'stimulates collagen'],
    ['centella asiatica',   'safe',         'sensitive,acne',     'healing, calming'],
    ['butylene glycol',     'safe',         'all',                'lightweight humectant'],
    ['squalene',            'safe',         'all',                'natural skin lipid'],
    ['dimethicone',         'neutral',      'all',                'silicone, non-comedogenic'],
    ['cyclomethicone',      'neutral',      'all',                'lightweight silicone'],
    ['alcohol denat',       'harmful',      'dry,sensitive',      'strips moisture barrier'],
    ['denatured alcohol',   'harmful',      'dry,sensitive',      'drying and irritating'],
    ['sd alcohol',          'harmful',      'dry,sensitive',      'similar to denatured alcohol'],
    ['fragrance',           'harmful',      'sensitive',          'common allergen'],
    ['parfum',              'harmful',      'sensitive',          'fragrance (French name)'],
    ['sodium lauryl sulfate','harmful',     'dry,sensitive',      'harsh cleanser'],
    ['sodium laureth sulfate','harmful',    'sensitive',          'milder SLS, can irritate'],
    ['essential oils',      'harmful',      'sensitive',          'natural but can irritate'],
    ['parabens',            'harmful',      'sensitive',          'preservative allergen'],
    ['coconut oil',         'comedogenic',  'acne,oily',          'high pore-clogging rating'],
    ['isopropyl myristate', 'comedogenic',  'acne',               'very high comedogenic'],
    ['lanolin',             'comedogenic',  'acne',               'thick, clogs pores'],
    ['cocoa butter',        'comedogenic',  'acne,oily',          'too rich for acne skin'],
    ['wheat germ oil',      'comedogenic',  'acne',               'high comedogenic rating'],
    ['algae extract',       'comedogenic',  'acne',               'may cause breakouts'],
]

df_ing = pd.DataFrame(data, columns=['ingredient', 'safety', 'suitable_for', 'description'])
df_ing.to_csv('ingredient_classification.csv', index=False)

print(f' Ingredient table: {len(df_ing)} entries')
print(df_ing['safety'].value_counts().to_string())
df_ing.head(8)


## Cell 5 Skin quiz
> User answers questions. Answers are stored in a `SkinProfile` object (OOP).

In [ ]:
# OOP: SkinProfile class
class SkinProfile:
    """
    Stores user skin type, concerns, city.
    OOP concepts: __init__, encapsulation, @property, @staticmethod, @classmethod
    """
    VALID_SKIN_TYPES = ['dry', 'oily', 'combination', 'sensitive', 'normal']

    def __init__(self, skin_type: str, concerns: list, city: str):
        self.__skin_type = skin_type.lower().strip()
        self.__concerns  = concerns
        self.__city      = city.strip()

    @property
    def skin_type(self):
      return self.__skin_type
    @property
    def concerns(self):
      return self.__concerns
    @property
    def city(self):
       return self.__city

    def get_profile(self) -> dict:
        return {'skin_type': self.__skin_type, 'concerns': self.__concerns, 'city': self.__city}

    @staticmethod
    def valid_types() -> list:
        return SkinProfile.VALID_SKIN_TYPES

    @classmethod
    def from_dict(cls, d: dict):
        return cls(d['skin_type'], d.get('concerns', []), d.get('city', 'Delhi'))

    def __repr__(self):
        return f'SkinProfile(type={self.__skin_type}, city={self.__city})'


#  Concern options
CONCERN_OPTIONS = [
    'acne / breakouts', 'dark spots', 'dullness',
    'large pores', 'redness / irritation', 'anti-aging',
    'dryness / dehydration', 'sun damage',
]

#  Quiz
def run_quiz() -> SkinProfile:
    print('=' * 50)
    print('       SKINSENSE — SKIN QUIZ')
    print('=' * 50)

    # Q1: Skin type
    print('\nQ1. What is your skin type?')
    for i, st in enumerate(SkinProfile.valid_types(), 1):
        print(f'    {i}. {st.title()}')
    while True:
        try:
            c = int(input('\nEnter number (1-5): ').strip())
            if 1 <= c <= 5:
                skin_type = SkinProfile.valid_types()[c - 1]
                break
            print('   Enter a number between 1 and 5.')
        except ValueError:
            print('   Please type a number.')

    # Q2: Concerns
    print('\nQ2. Select skin concerns (comma-separated numbers, or Enter to skip):')
    for i, c in enumerate(CONCERN_OPTIONS, 1):
        print(f'    {i}. {c}')
    concerns = []
    raw = input('\nYour choices: ').strip()
    if raw:
        for part in raw.split(','):
            try:
                idx = int(part.strip()) - 1
                if 0 <= idx < len(CONCERN_OPTIONS):
                    concerns.append(CONCERN_OPTIONS[idx])
            except ValueError:
                pass

    # Q3: City mapped to whole India
    print('\nQ3. Select your city:')
    city_list = list(INDIA_CITIES.keys())
    for i, c in enumerate(city_list, 1):
        print(f'    {i:2}. {c}')
    print(f'     0. Other (type city name manually)')

    city = 'Delhi'
    raw_city = input(f'\nEnter number (1-{len(city_list)}) or 0 for other: ').strip()
    try:
        idx = int(raw_city)
        if 1 <= idx <= len(city_list):
            city = city_list[idx - 1]
        elif idx == 0:
            city = input('   Type your city name: ').strip() or 'Delhi'
    except ValueError:
        city = raw_city if raw_city else 'Delhi'

    profile = SkinProfile(skin_type=skin_type, concerns=concerns, city=city)

    print('\n' + '-' * 50)
    print('Your skin profile:')
    print(f'   Skin type : {profile.skin_type.title()}')
    print(f'   Concerns  : {", ".join(profile.concerns) if profile.concerns else "None"}')
    print(f'   City      : {profile.city}')
    print('-' * 50)
    return profile


skin_profile = run_quiz()



## Cell 6 — Fetching weather data from  https://openweathermap.org/api using the api key

In [ ]:
import requests, json

WEATHER_API_KEY = input("Please enter your OpenWeatherMap API Key: ")

# ─── WeatherData class
class WeatherData:
    """OOP: class methods, __repr__, offline fallback from INDIA_CITIES dict."""

    def __init__(self, city, temperature, humidity, description, clouds, aqi=0):
        self.city        = city
        self.temperature = temperature
        self.humidity    = humidity
        self.description = description
        self.clouds      = clouds
        self.aqi         = aqi
        # Calculates real-time responsive UV estimate based on cloud cover density
        self.uv_estimate = max(1, round(10 - clouds / 10))

    @classmethod
    def from_api(cls, city: str, api_key: str):
        weather_url = 'https://api.openweathermap.org/data/2.5/weather'
        params = {'q': city, 'appid': api_key, 'units': 'metric'}
        try:
            # 1. Fetch Core Weather Metrics
            r = requests.get(weather_url, params=params, timeout=10)
            r.raise_for_status()
            w_data = r.json()

            # Extract coordinates returned from the weather query
            lat = w_data['coord']['lat']
            lon = w_data['coord']['lon']

            # 2. Fetch Live Air Pollution (AQI) using coordinates and the same key
            aqi_val = 0
            try:
                pollution_url = 'https://api.openweathermap.org/data/2.5/air_pollution'
                p_params = {'lat': lat, 'lon': lon, 'appid': api_key}
                pr = requests.get(pollution_url, params=p_params, timeout=5)
                if pr.status_code == 200:
                    p_data = pr.json()
                    # OpenWeather AQI scales from 1 (Good) to 5 (Very Poor)
                    raw_aqi = p_data['list'][0]['main']['aqi']

                    # Map qualitative index (1-5) into common target values for your dashboard
                    aqi_map = {1: 45, 2: 75, 3: 120, 4: 180, 5: 250}
                    aqi_val = aqi_map.get(raw_aqi, 100)
            except Exception:
                # Fallback to local INDIA_CITIES mapping values if pollution endpoint fails
                aqi_val = INDIA_CITIES.get(city, INDIA_CITIES['Delhi']).get('aqi', 90)

            # Ensure all 6 required positioning elements pass cleanly to instantiation
            return cls(
                city        = w_data.get('name', city),
                temperature = round(w_data['main']['temp'], 1),
                humidity    = w_data['main']['humidity'],
                description = w_data['weather'][0]['description'].title(),
                clouds      = w_data['clouds']['all'],
                aqi         = aqi_val
            )
        except requests.exceptions.HTTPError:
            print('API error — check key or city name. Using offline profile.')
            return cls.from_india_map(city)
        except Exception as e:
            print(f'Could not reach API ({e}). Using offline profile.')
            return cls.from_india_map(city)

    @classmethod
    def from_india_map(cls, city: str):
        """Uses INDIA_CITIES dict for 18 Indian cities — no API key needed."""
        profile = INDIA_CITIES.get(city, INDIA_CITIES['Delhi'])
        print(f'   Using offline weather profile for {city}')
        return cls(
            city        = city,
            temperature = profile['temperature'],
            humidity    = profile['humidity'],
            description = profile['description'],
            clouds      = max(0, 100 - profile['uv_estimate'] * 10),
            aqi         = profile.get('aqi', 0),
        )

    def to_dict(self): return vars(self)

    def __repr__(self):
        return (f'WeatherData({self.city}: {self.temperature}C, '
                f'hum={self.humidity}%, uv~{self.uv_estimate}, aqi={self.aqi})')


# Fetch weather
if not WEATHER_API_KEY:
    print('No API key — using offline India city profile')
    weather = WeatherData.from_india_map(skin_profile.city)
else:
    weather = WeatherData.from_api(city=skin_profile.city, api_key=WEATHER_API_KEY)

print(f'\nWeather: {weather}')

with open('weather_data.json', 'w') as f:
    json.dump(weather.to_dict(), f, indent=2)
print('Saved to weather_data.json')


## Cell 7 Recommendation engine
> Combines skin type + weather to score products and generate ingredient warnings.

In [ ]:
# RecommendationEngine class (OOP)
class RecommendationEngine:
    """
    Core logic. Takes SkinProfile + WeatherData + product DataFrame.
    Demonstrates: composition, static methods, regular methods.
    """

    # Ingredients to recommend per skin type
    GOOD_FOR = {
        'dry'        : ['hyaluronic acid', 'ceramide', 'glycerin', 'squalane',
                        'panthenol', 'shea butter', 'tocopherol'],
        'oily'       : ['niacinamide', 'salicylic acid', 'glycolic acid',
                        'aloe vera', 'zinc oxide'],
        'combination': ['niacinamide', 'hyaluronic acid', 'squalane', 'glycerin'],
        'sensitive'  : ['ceramide', 'aloe vera', 'panthenol', 'centella asiatica',
                        'zinc oxide', 'glycerin'],
        'normal'     : ['hyaluronic acid', 'glycerin', 'squalane', 'tocopherol'],
    }

    # Ingredient warning rules: (condition_fn, warning_message)
    WARNING_RULES = [
        (lambda ing, w, s: w.uv_estimate >= 6 and 'retinol' in ing,
         '  Retinol found — use at night only, causes photosensitivity in UV'),
        (lambda ing, w, s: w.humidity > 75 and any(x in ing for x in ['coconut oil','cocoa butter']),
         '  Heavy oils found — may feel greasy and clog pores in high humidity'),
        (lambda ing, w, s: 'fragrance' in ing and s == 'sensitive',
         '  Fragrance detected — known irritant for sensitive skin, patch test first'),
        (lambda ing, w, s: 'parfum' in ing and s == 'sensitive',
         '  Parfum detected — same as fragrance, risky for sensitive skin'),
        (lambda ing, w, s: w.temperature < 10 and 'alcohol denat' in ing,
         '  Denatured alcohol — very drying in cold weather'),
        (lambda ing, w, s: w.uv_estimate >= 8 and 'spf' not in ing and 'zinc oxide' not in ing,
         '  UV is very high today — apply SPF 50+ separately'),
        (lambda ing, w, s: s == 'oily' and 'coconut oil' in ing,
         '  Coconut oil — highly comedogenic, may worsen oily/acne skin'),
    ]

    def __init__(self, profile: SkinProfile, weather: WeatherData, df_products, df_ingredients):
        self.profile      = profile
        self.weather      = weather
        self.df_products  = df_products
        self.df_ing_table = df_ingredients

    def score_product(self, ingredients: list) -> int:
        """Score a product 0-10 based on how well it suits skin type + weather."""
        score = 0
        skin  = self.profile.skin_type
        good  = self.GOOD_FOR.get(skin, [])
        ing_str = ' '.join(ingredients)

        # +2 for each beneficial ingredient present
        for g in good:
            if g in ing_str:
                score += 2

        # Weather bonuses
        if self.weather.uv_estimate >= 6 and 'zinc oxide' in ing_str:
            score += 3
        if self.weather.humidity > 70 and any(x in ing_str for x in ['aloe', 'hyaluronic', 'glycerin']):
            score += 2
        if self.weather.temperature < 15 and any(x in ing_str for x in ['ceramide', 'squalane', 'shea']):
            score += 2

        # Penalties
        if 'fragrance' in ing_str and skin == 'sensitive':
            score -= 3
        if 'coconut oil' in ing_str and skin in ['oily', 'acne']:
            score -= 2
        if 'alcohol denat' in ing_str and skin == 'dry':
            score -= 2

        return max(0, score)

    def get_warnings(self, ingredients: list) -> list:
        """Return list of warnings for a product's ingredient list."""
        ing_str  = ' '.join(ingredients)
        skin     = self.profile.skin_type
        weather  = self.weather
        warnings = []
        for rule_fn, msg in self.WARNING_RULES:
            try:
                if rule_fn(ing_str, weather, skin):
                    warnings.append(msg)
            except Exception:
                continue
        return warnings

    def recommend(self, top_n: int = 6) -> pd.DataFrame:
        """Score all products and return top_n as DataFrame."""
        results = []
        for _, row in self.df_products.iterrows():
            ing_list = row.get('ingredients_clean', [])
            if not ing_list:
                continue
            score    = self.score_product(ing_list)
            warnings = self.get_warnings(ing_list)
            results.append({
                'name'       : row['name'],
                'brand'      : row.get('brand', ''),
                'score'      : score,
                'ingredients': row.get('ingredients_str', '')[:120],
                'warnings'   : ' | '.join(warnings) if warnings else 'None',
            })

        result_df = pd.DataFrame(results)
        result_df = result_df.sort_values('score', ascending=False).head(top_n)
        return result_df.reset_index(drop=True)

    @staticmethod
    def weather_tips(weather: WeatherData) -> list:
        """Static method: general skin tips based on today's weather."""
        tips = []
        if weather.uv_estimate >= 8:
            tips.append(' Very high UV — apply SPF 50+ every 2 hours')
        elif weather.uv_estimate >= 6:
            tips.append('  Moderate-high UV — SPF 30+ recommended')
        if weather.humidity < 30:
            tips.append(' Very dry air — apply moisturiser within 60s of washing face')
        if weather.humidity > 80:
            tips.append(' High humidity — swap heavy creams for lightweight gels')
        if weather.temperature < 10:
            tips.append(' Cold weather — use a barrier cream to prevent moisture loss')
        if weather.temperature > 35:
            tips.append(' Very hot — stay hydrated and use mattifying SPF if oily')
        if not tips:
            tips.append(' Weather conditions are comfortable for your skin today')
        return tips


# Run the engine
engine     = RecommendationEngine(skin_profile, weather, df, df_ing)
results_df = engine.recommend(top_n=6)
tips       = RecommendationEngine.weather_tips(weather)

print(f' Engine complete — {len(results_df)} products recommended')
results_df


## Cell 8 — Displaying the results as dashboard


In [ ]:
from IPython.display import display, HTML
import json
import os

def render_dashboard(profile, weather, results_df, tips):
    # 1. Convert recommendations and tips to clean JSON strings for JS consumption
    prods_json = results_df[['name','brand','score','ingredients','warnings']].to_json(orient='records')
    tips_json  = json.dumps(tips)

    # 2. Extract dynamic weather metrics from your Python weather object
    current_temp = weather.temperature
    current_hum  = weather.humidity
    current_uv   = weather.uv_estimate
    current_aqi  = weather.aqi
    current_desc = weather.description
    current_city = weather.city
    current_skin = profile.skin_type.title()

    # Pre-select chips matching the user's explicit profile concerns
    user_concerns = [c.split(' / ')[0].lower() for c in profile.concerns]

    # Loop directly over INDIA_CITIES list
    city_buttons = ''.join(
        f'<div class="city-btn {"sel" if c == current_city else ""}" style="pointer-events: none;">{c}</div>'
        for c in INDIA_CITIES
    )

    # JavaScript script-tag handling safely inside Python f-strings
    script_open  = '<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js">'
    script_close = '</script>'

    css = """
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: system-ui, sans-serif; background: #f7f3ef; color: #2d2d2d; font-size: 14px; }
  header { background:#fff; border-bottom:1px solid #e8e0d8; padding:14px 24px; display:flex; align-items:center; justify-content:space-between; }
  header h1 { font-size:18px; font-weight:600; color:#534AB7; }
  header span { font-size:12px; color:#888; }
  .main { max-width:960px; margin:0 auto; padding:24px 16px; }
  .step-nav { display:flex; align-items:center; margin-bottom:24px; }
  .dot { width:30px; height:30px; border-radius:50%; display:flex; align-items:center; justify-content:center; font-size:12px; font-weight:500; border:1.5px solid #ddd; background:#fff; color:#888; transition:all .2s; cursor:pointer; flex-shrink:0; }
  .dot.active { background:#534AB7; color:#fff; border-color:#534AB7; }
  .dot.done { background:#E1F5EE; color:#085041; border-color:#1D9E75; }
  .step-label { font-size:11px; color:#888; margin:0 10px 0 6px; white-space:nowrap; }
  .step-line { flex:1; height:1px; background:#ddd; }
  .panel { display:none; }
  .panel.show { display:block; }
  .card { background:#fff; border-radius:12px; border:1px solid #e8e0d8; padding:18px 20px; margin-bottom:14px; }
  .section-title { font-size:13px; font-weight:600; color:#444; margin-bottom:10px; }
  label { font-size:12px; color:#666; display:block; margin-bottom:5px; margin-top:12px; }
  .quiz-value { font-size:14px; font-weight:600; color:#534AB7; background:#eeeefa; padding:8px 12px; border-radius:6px; margin-bottom:10px; display:inline-block; }
  .chip-grid { display:flex; flex-wrap:wrap; gap:7px; margin-top:6px; }
  .chip { padding:5px 13px; font-size:12px; border:1px solid #ddd; border-radius:20px; background:#fafafa; color:#555; }
  .chip.on { background:#534AB7; color:#fff; border-color:#534AB7; }
  .city-grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(108px,1fr)); gap:7px; margin-top:8px; }
  .city-btn { padding:8px 10px; font-size:12px; text-align:center; border:1px solid #ddd; border-radius:8px; background:#fafafa; color:#555; opacity: 0.6; }
  .city-btn.sel { background:#E1F5EE; color:#085041; border-color:#1D9E75; font-weight:600; opacity: 1; }
  .btn { padding:9px 22px; font-size:13px; font-weight:500; border:1px solid #ddd; border-radius:9px; cursor:pointer; background:#fff; color:#2d2d2d; transition:background .15s; }
  .btn:hover { background:#f3f3f3; }
  .btn-primary { background:#534AB7; color:#fff; border-color:#534AB7; }
  .btn-primary:hover { background:#3C3489; }
  .btn-row { display:flex; gap:10px; margin-top:16px; }
  .metric-row { display:flex; gap:10px; flex-wrap:wrap; margin-bottom:14px; }
  .metric { flex:1; min-width:80px; background:#f7f3ef; border-radius:10px; padding:10px 14px; }
  .metric .val { font-size:20px; font-weight:600; color:#534AB7; }
  .metric .lbl { font-size:11px; color:#888; margin-top:2px; }
  .tip-line { font-size:12px; padding:8px 12px; border-left:3px solid #1D9E75; background:#E1F5EE; color:#085041; margin:4px 0; border-radius:0 7px 7px 0; }
  .prod-card { border:1px solid #e8e0d8; border-radius:11px; padding:14px 16px; margin-bottom:9px; background:#fff; }
  .prod-name { font-size:14px; font-weight:600; color:#2d2d2d; margin-bottom:2px; }
  .prod-brand { font-size:12px; color:#999; margin-bottom:8px; }
  .bar-bg { background:#eee; border-radius:4px; height:8px; margin:4px 0 8px; }
  .bar-fill { background:#534AB7; border-radius:4px; height:8px; transition:width .6s; }
  .warn-pill { display:inline-block; background:#FAEEDA; color:#633806; font-size:11px; padding:2px 9px; border-radius:20px; margin:2px 2px 2px 0; }
  .ing-text { font-size:11px; color:#888; margin-bottom:6px; }
  .chart-card { background:#fff; border-radius:12px; border:1px solid #e8e0d8; padding:18px 20px; margin-bottom:14px; }
"""

    js = f"""
const PRODS = {prods_json};
const TIPS  = {tips_json};
let barChart = null;

function goStep(n) {{
  [1,2,3,4].forEach(i => {{
    document.getElementById('p'+i).classList.toggle('show', i===n);
    const d = document.getElementById('d'+i);
    d.classList.remove('active','done');
    if (i===n) d.classList.add('active');
    else if (i<n) d.classList.add('done');
  }});
  if (n===3) renderProds();
  if (n===4) renderCharts();
}}

function renderProds() {{
  if(PRODS.length === 0) {{
    document.getElementById('prod-list').innerHTML = '<div class="card">No products matched this specific weather/skin filter combination. Try adjusting constraints.</div>';
    return;
  }}
  const maxScore = Math.max(...PRODS.map(p => p.score), 1);
  document.getElementById('prod-list').innerHTML = PRODS.map(p => {{
    const pct  = Math.round((p.score / maxScore) * 100);
    const warn = (p.warnings && p.warnings !== 'None')
      ? p.warnings.split(' | ').map(w => '<span class="warn-pill">'+w+'</span>').join('')
      : '';
    return '<div class="prod-card">'
      + '<div class="prod-name">'+p.name+'</div>'
      + '<div class="prod-brand">'+(p.brand||'Brand unknown')+'</div>'
      + '<div style="font-size:11px;color:#888">Match score: '+p.score+'</div>'
      + '<div class="bar-bg"><div class="bar-fill" style="width:'+pct+'%\"></div></div>'
      + '<div class="ing-text">'+p.ingredients+'</div>'
      + warn
      + '</div>';
  }}).join('');
}}

function renderCharts() {{
  if (barChart) {{ barChart.destroy(); barChart = null; document.getElementById('bar-canvas-wrap').innerHTML = ''; }}

  const h = Math.max(PRODS.length * 46 + 60, 220);
  const wrap = document.getElementById('bar-canvas-wrap');
  wrap.style.height = h + 'px';
  const canvas = document.createElement('canvas');
  wrap.appendChild(canvas);

  barChart = new Chart(canvas, {{
    type:'bar',
    data:{{
      labels: PRODS.map(p => p.name.length>35 ? p.name.slice(0,35)+'...' : p.name),
      datasets:[{{ label:'Match Score', data:PRODS.map(p=>p.score),
        backgroundColor:'#534AB7', borderWidth:0, borderRadius:4 }}]
    }},
    options:{{
      indexAxis:'y', responsive:true, maintainAspectRatio:false,
      scales:{{
        x:{{ min:0, max:12, ticks:{{ stepSize:2, color:'#888' }}, grid:{{ color:'rgba(0,0,0,.05)' }} }},
        y:{{ ticks:{{ color:'#888', font:{{ size:11 }} }}, grid:{{ display:false }} }}
      }},
      plugins:{{ legend:{{ display:false }} }}
    }}
  }});
}}

window.addEventListener('DOMContentLoaded', renderProds);
"""

    tips_html = ''.join(f'<div class="tip-line">{t}</div>' for t in tips)

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>SkinSense Dashboard</title>
{script_open}{script_close}
<style>{css}</style>
</head>
<body>

<header>
  <h1>SkinSense Dashboard</h1>
  <span>Beauty + Weather Advisor &middot; Prototype V2</span>
</header>

<div class="main">

  <div class="step-nav">
    <div class="dot active" id=\"d1\" onclick="goStep(1)">1</div>
    <span class="step-label">Your Profile</span>
    <div class="step-line"></div>
    <div class="dot" id="d2" onclick="goStep(2)">2</div>
    <span class="step-label">Live Weather</span>
    <div class="step-line"></div>
    <div class="dot" id="d3" onclick="goStep(3)">3</div>
    <span class="step-label">Recommended Products</span>
    <div class="step-line"></div>
    <div class="dot" id="d4" onclick="goStep(4)">4</div>
    <span class="step-label">Analytics Insights</span>
  </div>

  <div class="panel show" id="p1">
    <div class="card">
      <div class="section-title">Verified Skin Profile (from Python Quiz)</div>

      <label>Skin Type</label>
      <div class="quiz-value">{current_skin} Skin</div>

      <label>Target Concerns</label>
      <div class="chip-grid">
        <span class="chip {'on' if 'acne' in user_concerns or 'breakouts' in user_concerns else ''}">Acne / Breakouts</span>
        <span class="chip {'on' if 'dark spots' in user_concerns else ''}">Dark spots</span>
        <span class="chip {'on' if 'dullness' in user_concerns else ''}">Dullness</span>
        <span class="chip {'on' if 'large pores' in user_concerns else ''}">Large pores</span>
        <span class="chip {'on' if 'redness' in user_concerns or 'irritation' in user_concerns else ''}">Redness / Irritation</span>
        <span class="chip {'on' if 'anti-aging' in user_concerns else ''}">Anti-aging</span>
        <span class="chip {'on' if 'dryness' in user_concerns or 'dehydration' in user_concerns else ''}">Dryness / Dehydration</span>
        <span class="chip {'on' if 'sun damage' in user_concerns else ''}">Sun Damage</span>
      </div>

      <label>Evaluated Location</label>
      <div class="city-grid">{city_buttons}</div>
    </div>
    <button class="btn btn-primary" onclick="goStep(2)">View Local Weather Insights &rarr;</button>
  </div>

  <div class="panel" id="p2">
    <div class="card">
      <div class="section-title">{current_city} Environment Diagnostics</div>
      <div class="metric-row">
        <div class="metric"><div class="val">{current_temp}&deg;C</div><div class="lbl">Temperature</div></div>
        <div class="metric"><div class="val">{current_hum}%</div><div class="lbl">Humidity</div></div>
        <div class="metric"><div class="val">{current_uv}/10</div><div class="lbl">UV Index</div></div>
        <div class="metric"><div class="val">{current_aqi}</div><div class="lbl">AQI (Air Quality)</div></div>
        <div class="metric"><div class="val" style="font-size:13px">{current_desc}</div><div class="lbl">Condition Summary</div></div>
      </div>
      <div>{tips_html}</div>
    </div>
    <div class="btn-row">
      <button class="btn" onclick="goStep(1)">&larr; Review Profile</button>
      <button class="btn btn-primary" onclick="goStep(3)">Generate Product Scores &rarr;</button>
    </div>
  </div>

  <div class="panel" id="p3">
    <div class="section-title" style="margin-bottom: 12px;">Top Dynamic Product Recommendations for your conditions:</div>
    <div id="prod-list"></div>
    <div class="btn-row">
      <button class="btn" onclick="goStep(2)">&larr; Back to Environment</button>
      <button class="btn btn-primary" onclick="goStep(4)">See Analytics Charts &rarr;</button>
    </div>
  </div>

  <div class="panel" id="p4">
    <div class="chart-card">
      <div class="section-title" style="font-size:14px; margin-bottom:15px;">Top Product Match Evaluation</div>
      <div id="bar-canvas-wrap" style="position:relative; width:100%;"></div>
    </div>
    <div class="btn-row">
      <button class="btn" onclick="goStep(3)">&larr; Back to Products</button>
    </div>
  </div>

</div>

<script>{js}</script>
</body>
</html>"""

    # Save standalone HTML file
    filepath = 'skinsense_dashboard.html'
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(html)

    size_kb = round(os.path.getsize(filepath) / 1024, 1)
    print(f' Sync Complete! Saved cleaner dashboard to: {filepath} ({size_kb} KB)')

    # Show inline inside the notebook frame using srcdoc
    escaped = html.replace('"', '&quot;')
    display(HTML(f'<iframe srcdoc="{escaped}" width="100%" height="750" style="border:none;border-radius:12px;background:#f7f3ef;"></iframe>'))

render_dashboard(skin_profile, weather, results_df, tips)